# Lab_Ch04 — 學術檢索與 AI 對照核對 Notebook

**課程**：MSBA6125 管理資訊系統與科技
**主題**：Scopus 文獻檢索、開放取用（Open Access, OA）PDF 下載、AI 聊天機器人引用核對

本 notebook 對應 Lab_Ch04 指引之操作步驟，單元順序如下：

| 單元 | 對應步驟 | 內容 |
|---|---|---|
| 除錯練習 | 步驟 3 | 以 Colab 內建 Gemini 找出並修正程式錯誤 |
| 環境設定 | 步驟 4 | 自 Colab Secrets 載入 API 金鑰 |
| 第一輪檢索（寬泛） | 步驟 5 | Scopus 檢索前 10 筆，選 5 篇 |
| 第二輪檢索（聚焦） | 步驟 6 | 改寫查詢語法，窄化結果 |
| 開放取用查詢 | 步驟 7 | Unpaywall 逐篇查 OA 狀態 |
| 下載開放取用 PDF | 步驟 7 | 下載可合法取得之 PDF |
| AI 對照核對 | 步驟 8 | 核對聊天機器人引用之真實性 |
| 匯出結果 | 步驟 9 | 檢索結果存為 CSV |

> **重要**：本 notebook 不含任何金鑰。金鑰一律經 Colab Secrets 存取（見「環境設定」單元）；請勿將金鑰貼入任何單元。
>
> **本 Lab 之範圍**：本 Lab 僅示範以 API 連接 Scopus 資料庫取得論文書目資訊（含摘要）；檢索結果之研究缺口（research gap）分析須以代理式 AI（agentic AI）執行——屬後續課程主題。本 Lab 產出之書目與摘要資料即為該分析之輸入。

## 除錯練習（步驟 3）——以 Colab 內建 Gemini 修正程式錯誤

**本 notebook 含一處刻意設計之程式錯誤。** 請先依序（自上而下）執行部分單元，直到出現錯誤訊息——錯誤單元左側顯示紅色叉號。

除錯方式（AI 輔助除錯實作）：

1. 於 Colab 版面右側或左側邊欄點擊 **Gemini 圖示**（星形／Gemini 標誌）開啓 AI 助理面板
2. 將錯誤訊息貼入對話框並描述情境——示例：`執行這個單元時出現以下錯誤：<貼上錯誤訊息>。請找出原因並提供修正方式。`
3. 閱讀診斷與修正建議——**先理解錯在哪裡**，再依建議修改：將修正後之程式碼貼回原單元，或直接採用面板之差異檢視（diff view）套用修正
4. 重新執行該單元直至通過（綠色勾號）——**此為全部後續步驟之執行前提**

**VPN 提醒**：Gemini 於部分地區需經 VPN 方可使用；Colab 之 Gemini 助理同受此限——無法連線時開啓 VPN 再重試。

記錄：以一句話寫出錯誤原因與修正方式。

## 環境設定（步驟 4）

執行下方程式單元，自 Colab Secrets 載入兩項金鑰。**執行前先完成 Lab_Ch04 指引步驟 1（申請 Scopus API key）與步驟 4（存入 Secrets）**：
- `SCOPUS_API_KEY`：Elsevier Developer Portal 取得之個人 key
- `CONTACT_EMAIL`：你的學校郵箱（Unpaywall 查詢要求真實郵箱）

預期輸出：「金鑰已載入」——僅顯示成功訊息，不顯示金鑰內容。

In [ ]:
# 環境設定：自 Colab Secrets 載入金鑰（不顯示金鑰內容），並定義共用工具函式
try:
    from google.colab import userdata
    SCOPUS_API_KEY = userdata.get('SCOPUS_API_KEY')
    CONTACT_EMAIL = userdata.get('CONTACT_EMAIL')
except Exception:
    # 非 Colab 環境（如本機 Jupyter）：以環境變數替代
    import os
    SCOPUS_API_KEY = os.environ.get('SCOPUS_API_KEY', '')
    CONTACT_EMAIL = os.environ.get('CONTACT_EMAIL', '')

NOTEBOOK_READY = bool(SCOPUS_API_KEY)
if NOTEBOOK_READY:
    print('金鑰已載入')
else:
    print('未偵測到 SCOPUS_API_KEY——請依 Lab_Ch04 指引步驟 4 完成 Colab Secrets 設定後，重新執行本單元。')
    print('（未設定前，後續檢索單元將顯示此提示並跳過）')

all_results = []  # 檢索結果累積（供後續單元與匯出使用）

import requests  # 後續各單元共用

def scopus_search(query, count=10):
    """以 Scopus Search API 檢索，回傳書目清單（title/year/doi/journal/cited_by/abstract）。
    優先以 COMPLETE 檢視取得摘要；金鑰未含該檢視權限時自動退回 STANDARD（摘要留空）。"""
    params = {'query': query, 'count': count,
              'httpAccept': 'application/json', 'sort': 'citedby-count'}
    headers = {'X-ELS-APIKey': SCOPUS_API_KEY, 'Accept': 'application/json'}
    url = 'https://api.elsevier.com/content/search/scopus'
    resp = requests.get(url, params={**params, 'view': 'COMPLETE'},
                        headers=headers, timeout=60)
    if resp.status_code != 200:
        # 金鑰未含 COMPLETE 檢視時退回 STANDARD——書目資訊不受影響
        resp = requests.get(url, params={**params, 'view': 'STANDARD'},
                            headers=headers, timeout=60)
    if resp.status_code != 200:
        raise RuntimeError(f'Scopus API 錯誤（HTTP {resp.status_code}）——請確認 key 有效；401 = 未授權；429 = 配額')
    entries = resp.json().get('search-results', {}).get('entry', [])
    entries = [e for e in entries if 'error' not in e]  # 濾除空結果集之 error 項目
    return [{'title': e.get('dc:title', ''),
             'year': (e.get('prism:coverDate') or '')[:4],
             'doi': e.get('prism:doi', ''),
             'journal': e.get('prism:publicationName', ''),
             'cited_by': e.get('citedby-count', 0),
             'abstract': (e.get('dc:description') or '').strip()} for e in entries]

def unpaywall_check(doi):
    """查詢 Unpaywall；回傳 (is_oa, oa_url_for_pdf, version)。"""
    url = f'https://api.unpaywall.org/v2/{doi}'
    resp = requests.get(url, params={'email': CONTACT_EMAIL}, timeout=60)
    if resp.status_code == 422:
        raise RuntimeError('Unpaywall 拒絕郵箱格式（HTTP 422）——CONTACT_EMAIL 須為真實格式之郵箱')
    if resp.status_code != 200:
        return False, '', ''
    d = resp.json()
    best = d.get('best_oa_location') or {}
    return bool(d.get('is_oa')), best.get('url_for_pdf') or '', best.get('version') or ''


def verify_citation(title, year, doi):
    """以 Scopus 核對單一引用；回傳 (狀態, 資料庫紀錄標題)。
    狀態：已核實（DOI 存在且標題／年份一致）／資料不符（DOI 存在但書目不同）／查無此文／格式問題。"""
    import re
    from difflib import SequenceMatcher
    if not doi.strip():
        return '未提供 DOI', ''
    if not re.match(r'^10\.\d{4,9}/\S+$', doi):
        return 'DOI 格式無效', ''
    hits = scopus_search(f'DOI("{doi}")', count=1)
    if not hits:
        return '查無此文', ''
    db = hits[0]
    same_title = SequenceMatcher(None, title.lower().strip(),
                                 (db['title'] or '').lower().strip()).ratio() >= 0.6
    same_year = (not year.strip()) or (year.strip() == db['year'])
    if same_title and same_year:
        return '已核實', db['title']
    return '資料不符', db['title']


def download_pdf(url, doi):
    """下載 PDF 並驗證檔案內容為 PDF；回傳檔名或空字串。"""
    import os
    os.makedirs('downloads', exist_ok=True)
    fname = 'downloads/' + doi.replace('/', '_')[:120] + '.pdf'
    # 部分出版社會封鎖非瀏覽器請求——帶瀏覽器 User-Agent（2026-09-20 Gemini 除錯實測）
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                             '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        resp = requests.get(url, headers=headers, timeout=120, allow_redirects=True)
        if resp.status_code == 200 and (resp.content[:4] == b'%PDF'
                                        or resp.headers.get('Content-Type', '').startswith('application/pdf')):
            with open(fname, 'wb') as f:
                f.write(resp.content)
            return fname
        ct = resp.headers.get('Content-Type', '未知')
        print(f'  未下載（HTTP {resp.status_code}／Content-Type: {ct}）——{url}')
    except requests.exceptions.RequestException as e:
        print(f'  下載失敗（{type(e).__name__}）——{url}')
    return ''

## 第一輪檢索（寬泛）（步驟 5）

1. 將下行 `你的主題` 整體取代為你的檢索主題（示例：`digital transformation in SMEs`）
2. 執行本單元——檢索 Scopus 中 2023 年以後之文獻，輸出前 10 筆（標題、年份、DOI、被引次數、期刊與摘要；摘要於畫面顯示前 200 字元，全文存於匯出 CSV）
3. 閱讀標題與摘要，選出與你主題最相關的 5 篇並記錄編號

In [ ]:
# 第一輪檢索（寬泛）：修改下行主題文字後執行
你的主題 = 'digital transformation in SMEs'

if not NOTEBOOK_READY:
    print('未偵測到 SCOPUS_API_KEY——請先完成「環境設定」單元。')
else:
    broad_query = f'TITLE-ABS-KEY({你的主題}) AND PUBYEAR > 2022'
    broad_results = scopus_search(broad_query)
    all_results.extend(broad_results)

    print(f'檢索式：{broad_query}')
    print(f'取得 {len(broad_results)} 筆結果：\n')
    for i, r in enumerate(broad_results, 1):
        print(f"{i}. {r['title']}")
        print(f"   {r['year']} | 引用 {r['cited_by']} 次 | DOI: {r['doi'] or '（無）'}")
        ab = r['abstract']
        ab = ab[:200] + ('…' if len(ab) > 200 else '')
        print(f"   摘要：{ab or '（無摘要）'}")
        print(f"   期刊：{r['journal']}\n")

    if not broad_results:
        print('0 筆結果——請減少關鍵詞或改用更廣泛之同義詞後重新執行本單元。')
    elif not any(r['abstract'] for r in broad_results):
        print('註：本次未回傳摘要——您的金鑰可能未含 COMPLETE 檢視權限（書目資訊不受影響）。')

## 第二輪檢索（聚焦）（步驟 6）

1. 根據第一輪所選 5 篇之標題與摘要，歸納 1–2 組更精確之關鍵詞組合
2. 修改下行 `你的聚焦查詢` 之兩組關鍵詞後執行
3. 比較兩輪結果之數量與主題集中度——以一句話記錄「窄化後聚焦於哪個子議題」（填入下行設定處）

In [ ]:
# 第二輪檢索（聚焦）：修改主題關鍵詞組合後執行
# 範例：(generative AI OR large language model) AND (advertising OR consumer engagement)
你的聚焦查詢 = '(generative AI OR large language model) AND (advertising OR consumer engagement)'

if not NOTEBOOK_READY:
    print('未偵測到 SCOPUS_API_KEY——請先完成「環境設定」單元。')
else:
    focus_query = f'TITLE-ABS-KEY({你的聚焦查詢}) AND PUBYEAR > 2022'
    focus_results = scopus_search(focus_query, count=5)
    all_results.extend(focus_results)

    print(f'檢索式：{focus_query}')
    print(f'第一輪 {len(broad_results)} 筆 → 第二輪 {len(focus_results)} 筆\n')
    for i, r in enumerate(focus_results, 1):
        print(f"{i}. {r['title']}（{r['year']}）")
        ab = r['abstract']
        ab = ab[:200] + ('…' if len(ab) > 200 else '')
        print(f"   摘要：{ab or '（無摘要）'}")
        print(f"   DOI: {r['doi'] or '（無）'}\n")

    # 記錄：以一句話寫出窄化後聚焦於哪個子議題（下行預設值為硬編碼範例——示範用）
    子議題記錄 = '（範例）窄化後聚焦於生成式 AI 在廣告與消費者參與之應用'
    print(f'子議題記錄：{子議題記錄}')
    print()
    if '（範例）' in 子議題記錄:
        print('註：上述子議題記錄為硬編碼（hardcoded）內容——僅供示範輸出格式，非實際分析結果；')
        print('實際操作時，請自行閱讀上方檢索結果後改寫之。')
        print()
    print('程序說明：完整研究流程中，本步驟之結果（子議題歸納與聚焦檢索式）應以 AI 代理')
    print('（agentic AI）產出——由代理讀取第一輪檢索結果並歸納；本單元尚未涵蓋 AI 代理')
    print('操作（後續課程主題），故以手動流程示意。')

## 開放取用查詢（步驟 7）

執行本單元——notebook 以各篇之 DOI 逐一查詢 Unpaywall API，輸出開放取用（OA）狀態表：
- **可開放取用**：顯示 OA 版本類型（published / accepted / preprint）與授權
- **非開放取用**：顯示「僅保留書目資訊」——訂閱制論文之全文取得須經機構訂閱或圖書館服務

預期輸出：每篇顯示 OA 狀態；記錄可下載篇數與非 OA 篇數。

In [ ]:
# 開放取用（OA）查詢：以 Unpaywall API 逐篇查詢 DOI
if not NOTEBOOK_READY:
    print('未偵測到 SCOPUS_API_KEY——請先完成「環境設定」單元。')
elif not CONTACT_EMAIL:
    print('未偵測到 CONTACT_EMAIL——請回 Lab_Ch04 指引步驟 4 補設（Unpaywall 要求真實郵箱）。')
else:
    oa_count = 0
    seen = set()
    print('OA 狀態表：\n')
    for r in all_results:
        doi = r['doi']
        if not doi or doi in seen:
            continue
        seen.add(doi)
        r['is_oa'], r['oa_url_for_pdf'], r['oa_version'] = unpaywall_check(doi)
        if r['is_oa']:
            oa_count += 1
            print(f"[OA] {r['title'][:60]}｜版本：{r['oa_version'] or '未標示'}｜DOI: {doi}")
        else:
            print(f"[非 OA] {r['title'][:60]}｜僅保留書目資訊｜DOI: {doi}")

    print(f'\n統計：可開放取用 {oa_count} 篇；非 OA {len(seen) - oa_count} 篇（共 {len(seen)} 篇）')

## 下載開放取用 PDF（步驟 7）

執行本單元——將可開放取用（且有 PDF 連結）之論文下載至本 notebook 檔案區之 `downloads` 資料夾：
1. 執行下方的下載單元
2. 點擊 Colab 左側邊欄資料夾圖示（Files）查看，檔案位於 `downloads` 資料夾
3. 預期結果：OA 論文下載成功（檔名含 DOI）；個別下載失敗不影響其餘篇數

**說明**：本練習僅下載開放取用版本（作者／出版社授權公開之版本）——此即合法下載界線；訂閱制論文之全文取得須經機構訂閱或圖書館服務。

In [ ]:
# 下載可開放取用之 PDF 至 downloads 資料夾（僅限 OA 版本）
if not NOTEBOOK_READY:
    print('未偵測到 SCOPUS_API_KEY——請先完成「環境設定」單元。')
else:
    downloaded = []
    for r in all_results:
        if not r.get('is_oa') or not r.get('oa_url_for_pdf'):
            continue
        fname = download_pdf(r['oa_url_for_pdf'], r['doi'])
        if fname:
            downloaded.append(fname)
            print(f'已下載：{fname}')
        else:
            print(f"下載失敗（跳過，不影響其餘）：{r['title'][:50]}")

    print(f'\n本次下載 {len(downloaded)} 篇 PDF 至 downloads 資料夾。')
    if not downloaded:
        print('無成功下載——若所選皆非 OA：放寬主題重跑一輪，直至取得至少 1 篇。')

## AI 對照核對（步驟 8）——請先於瀏覽器完成聊天機器人提問

**執行本單元前**，先在瀏覽器向 AI 聊天機器人提問（Lab_Ch04 指引步驟 8 提供提示詞範例），取得 3 篇附標題／年份／DOI 之文獻。

將聊天機器人回覆之三項資訊，依下列格式填入下方設定處（一字不改照抄；缺 DOI 留空字串）：

```python
chatbot_titles = ['第一篇文章標題', '第二篇文章標題', '第三篇文章標題']
chatbot_years  = ['2024', '2023', '2024']
chatbot_dois   = ['10.xxxx/xxxxx', '', '10.yyyy/yyyyy']
```

執行本單元——notebook 將逐筆以文獻資料庫核對聊天機器人提供之 DOI，並輸出核對表與統計。

In [ ]:
# AI 對照核對：以文獻資料庫核對聊天機器人引用之真實性
# ——修改下行三個清單為聊天機器人回覆之內容後執行
chatbot_titles = ['（範例）Generative AI and the future of work', '（範例）Digital transformation in SMEs']
chatbot_years = ['2024', '2023']
chatbot_dois = ['10.xxxx/xxxxx', '']

if not NOTEBOOK_READY:
    print('未偵測到 SCOPUS_API_KEY——請先完成「環境設定」單元。')
elif not any(d.strip() for d in chatbot_dois):
    print('尚無可核對之 DOI——請先將聊天機器人回覆填入上方清單（含 DOI）後重新執行。')
else:
    print('聊天機器人引用核對表：\n')
    n_verified = n_mismatch = n_notfound = 0
    for t, y, d in zip(chatbot_titles, chatbot_years, chatbot_dois):
        status, db_title = verify_citation(t, y, d)
        if status == '已核實':
            n_verified += 1
        elif status == '資料不符':
            n_mismatch += 1
        elif status == '查無此文':
            n_notfound += 1
        print(f'· {t}（{y}）｜DOI: {d or "（無）"}')
        print(f'  → {status}' + (f'｜資料庫標題：{db_title[:60]}' if db_title else '') + '\n')

    total = len(chatbot_titles)
    print(f'統計：{total} 篇中——已核實 {n_verified} 篇；資料不符 {n_mismatch} 篇；查無此文 {n_notfound} 篇。')
    print('說明：「查無此文」可能表示虛構引用（hallucinated citation）——AI 生成之引用一律須經資料庫核實。')

## 匯出結果（步驟 9）

執行本單元——將兩輪檢索結果（含摘要全文）存為 CSV 檔（檔名含你的主題），並顯示檔案下載方式。

1. 執行下方的匯出單元
2. 於 Colab 左側邊欄檔案區下載 CSV（或點擊輸出之連結）保存
3. 後續依 Lab_Ch04 指引步驟 9 完成 Scopus 網頁人工核對

最後更新 `downloads`、`Lab_Ch04_檢索結果_*.csv` 之狀態並保留個人副本。

In [ ]:
# 匯出檢索結果為 CSV
if not NOTEBOOK_READY:
    print('未偵測到 SCOPUS_API_KEY——請先完成「環境設定」單元。')
elif not all_results:
    print('尚無檢索結果——請先完成第一輪／第二輪檢索單元。')
else:
    import csv

    fname = f"Lab_Ch04_檢索結果_{你的主題.replace(' ', '_')}.csv"
    fields = ['title', 'year', 'doi', 'journal', 'cited_by', 'abstract', 'is_oa', 'oa_version']
    with open(fname, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        for r in all_results:
            writer.writerow({k: r.get(k, '') for k in fields})

    print(f'已匯出：{fname}')
    print('下載方式：點擊 Colab 左側邊欄資料夾圖示（Files）→ 找到本檔 → 點擊 ⋮ → Download。')